Imports and paths

In [1]:
import os
import pandas as pd
from icecream import ic
from pathlib import Path
from pandas import DataFrame

THIS_FILE_PATH = Path(os.getcwd()) / "olistbr.ipynb"

THIS_PROJECT_PATH = THIS_FILE_PATH.parent.parent
DATA_RAW_PATH = THIS_PROJECT_PATH / "data" / "raw"

if not DATA_RAW_PATH.exists():
    raise FileExistsError()

ic(THIS_PROJECT_PATH)
ic(THIS_FILE_PATH)
ic(DATA_RAW_PATH)

ic| THIS_PROJECT_PATH: WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr')
ic| THIS_FILE_PATH: WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr/notebooks/olistbr.ipynb')
ic| DATA_RAW_PATH: WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr/data/raw')


WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr/data/raw')

In [2]:
raw_filenames: list[str] = os.listdir(DATA_RAW_PATH)
raw_filepaths: list[Path] = [DATA_RAW_PATH / f for f in raw_filenames]
# ic(raw_filenames)
# ic(raw_filepaths)

dfs: dict[str, DataFrame] = {f.name: pd.read_csv(f, dtype=str) for f in raw_filepaths}
# ic(dfs)

In [33]:
import string


def infer_dtypes(df: DataFrame) -> DataFrame:
    """
    df -> out_df

    ascii: 0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~
    if any entry in col is not in ascii, then use national ('n' in 'nvarchar')

    df rows start at 0


    """

    def is_chars_in_string(chars: str, parent_string: str) -> bool:
        if set(chars).intersection(set(parent_string)):
            return True

        return False

    out_df = pd.DataFrame(
        index=df.columns,
        columns=[
            "has_nulls",
            "how_many_nulls",
            "where_nulls",
            "sorted_chars_used",
            "has_digits",
            "has_decimal",
            "has_dash",
            "has_colon",
            "has_space",
            "has_ascii",
            "has_non_ascii",
            "has_prefix_zero",
            "entry_lengths",
            "max_entry_length",
            "is_fixed_length",
            "has_unique_entries",
            "has_exactly_two_entries",
        ],
    )

    cols = df.columns

    for col in cols:
        print(col)

        where_null = df[col].isnull()

        out_df.loc[col, "where_nulls"] = df[where_null].index.tolist()
        out_df.loc[col, "has_nulls"] = 1 if where_null.any() else 0
        out_df.loc[col, "how_many_nulls"] = where_null.sum()

        clean_series = df[col].dropna()

        chars_used: set[str] = set("".join(clean_series.astype(str)))
        sorted_chars_used: str = "".join(sorted(chars_used))
        out_df.loc[col, "sorted_chars_used"] = sorted_chars_used

        out_df.loc[col, "has_digits"] = (
            1 if is_chars_in_string(string.digits, sorted_chars_used) else 0
        )
        out_df.loc[col, "has_decimal"] = (
            1 if is_chars_in_string(".", sorted_chars_used) else 0
        )
        out_df.loc[col, "has_dash"] = (
            1 if is_chars_in_string("-", sorted_chars_used) else 0
        )
        out_df.loc[col, "has_colon"] = (
            1 if is_chars_in_string(":", sorted_chars_used) else 0
        )
        out_df.loc[col, "has_space"] = (
            1 if is_chars_in_string(" ", sorted_chars_used) else 0
        )
        out_df.loc[col, "has_ascii"] = (
            1 if is_chars_in_string(string.printable, sorted_chars_used) else 0
        )

        # if col chars > ascii when col chars - ascii > 0
        set_minus = chars_used - set(string.printable)
        out_df.loc[col, "has_non_ascii"] = 1 if set_minus else 0

        unique_entries = df[col].unique()

        out_df.loc[col, "has_prefix_zero"] = (
            1 if clean_series.astype(str).str.startswith("0").any() else 0
        )
        entry_lengths = sorted(clean_series.astype(str).str.len().unique().tolist())

        out_df.loc[col, "entry_lengths"] = entry_lengths
        out_df.loc[col, "max_entry_length"] = max(entry_lengths)
        out_df.loc[col, "is_fixed_length"] = 1 if len(entry_lengths) == 1 else 0
        out_df.loc[col, "has_unique_entries"] = 1 if df[col].is_unique else 0

    return out_df


filename = "olist_closed_deals_dataset.csv"
df = dfs[filename]
infer_dtypes(df)

mql_id
seller_id
sdr_id
sr_id
won_date
business_segment
lead_type
lead_behaviour_profile
has_company
has_gtin
average_stock
business_type
declared_product_catalog_size
declared_monthly_revenue


,has_nulls,how_many_nulls,where_nulls,sorted_chars_used,has_digits,has_decimal,has_dash,has_colon,has_space,has_ascii,has_non_ascii,has_prefix_zero,entry_lengths,max_entry_length,is_fixed_length,has_unique_entries,has_exactly_two_entries
mql_id,0,0,[],0123456789abcdef,1,0,0,0,0,1,0,1,[32],32,1,1,NaN
seller_id,0,0,[],0123456789abcdef,1,0,0,0,0,1,0,1,[32],32,1,1,NaN
sdr_id,0,0,[],0123456789abcdef,1,0,0,0,0,1,0,1,[32],32,1,0,NaN
sr_id,0,0,[],0123456789abcdef,1,0,0,0,0,1,0,1,[32],32,1,0,NaN
won_date,0,0,[],-0123456789:,1,0,1,1,1,1,0,0,[19],19,1,1,NaN
business_segment,0,0,[],_abcdefghiklmnoprstuvwy,0,0,0,0,0,1,0,0,"[3, 4, 5, 7, 9, 10, 11, 12, 13, 14, 15, 16, 19...",31,0,0,NaN
lead_type,0,0,[],_abdefghilmnorstuy,0,0,0,0,0,1,0,0,"[5, 7, 8, 10, 12, 13, 15]",15,0,0,NaN
lead_behaviour_profile,0,0,[],",acefghklorstw",0,0,0,0,1,1,0,0,"[3, 4, 5, 9, 10, 11]",11,0,0,NaN
has_company,0,0,[],FTaelrsu,0,0,0,0,0,1,0,0,"[4, 5]",5,0,0,NaN
has_gtin,1,2,"[34, 745]",FTaelrsu,0,0,0,0,0,1,0,0,"[4, 5]",5,0,0,NaN


In [35]:
filename = "olist_closed_deals_dataset.csv"
col = "declared_monthly_revenue"

df = dfs[filename]

print(df[col])

df_clean = df[col].dropna()
# print(df_clean)

print(df_clean.unique())
print(type(df))
print(type(df_clean))

30          0.0
34          0.0
41       6000.0
70      30000.0
86          0.0
         ...   
780     25000.0
787     50000.0
810    120000.0
811         0.0
813    250000.0
Name: declared_monthly_revenue, Length: 63, dtype: object
['0.0' '6000.0' '30000.0' '150000.0' '25000.0' '100000.0' '210000.0'
 '15000.0' '20000.0' '250000.0' '8000000.0' '5000.0' '4000.0' '10000.0'
 '300000.0' '60000.0' '1000.0' '50000.0' '130000.0' '120000.0' '8000.0']
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>


In [5]:
# scratch work

chars = "ab0z"
parent_string = string.ascii_letters

print(set(chars))
print(set(parent_string))


# print(dfs)
# print(dfs["olist_order_reviews_dataset.csv"]['review_comment_title'].unique().tolist())

{'a', '0', 'b', 'z'}
{'Q', 't', 'z', 'i', 'v', 'M', 'f', 'w', 'D', 'm', 'y', 'O', 'n', 'q', 'U', 'I', 'k', 'g', 'A', 'T', 'c', 'r', 'Y', 'V', 'S', 'R', 'b', 'd', 'G', 'a', 'W', 'J', 'u', 'x', 'o', 'K', 'L', 'N', 'Z', 'X', 'E', 'p', 'F', 's', 'H', 'e', 'h', 'B', 'l', 'j', 'C', 'P'}


In [6]:
"hello"

'hello'

In [7]:
months = {
    "january",
    "february",
    "march",
    "april",
    "may",
    "june",
    "july",
    "august",
    "september",
    "october",
    "november",
    "december",
}

print(sorted(set("".join(string.ascii_lowercase)).difference(set("".join(months)))))

['k', 'q', 'w', 'x', 'z']


In [8]:
# dfs.keys()


# def describe_table(current_table: str, current_column: str) -> None:
#     df = dfs[current_table]

#     print(f"current column: {current_column}")
#     print("")
#     print(df[current_column].sample(5))
#     print("")
#     print(f"dtype:  {df[current_column].dtype}")
#     print(f"table:  {current_table}")
#     print(f"column: {current_column}")
#     print("")

#     # check if contains null
#     has_null = df[current_column].isnull().any()
#     print(f"contains NULL:  {has_null}")

#     # if has_null:
#     #     # filter out null
#     #     print("contains null")
#     #     df_notnull = df[df[current_column].notnull()]
#     #     pass

#     if df[current_column].dtype != "object":
#         print("not an object")

#         df[current_column] = df[current_column].dropna()

#         print(df[current_column].describe())

#         return

#     # check if requires 'n' prefix: nchar, nvarchar
#     # n = national, means contains unicode
#     is_national: bool = not df[current_column].apply(lambda x: str(x).isascii()).all()

#     # check if char or varchar
#     entry_lengths = df[current_column].dropna().str.len().unique()

#     max_length = entry_lengths.max()
#     is_fixed_length = entry_lengths.size == 1

#     # print(f'is char:        {is_char}')
#     print(f"is above ascii: {is_national}")
#     print(f"entry lengths:  {entry_lengths}")
#     print(f"max length:     {max_length}")
#     print(f"fixed length:   {is_fixed_length}")

# table = "olist_geolocation_dataset.csv"
# column = "geolocation_state"
# describe_table(current_table=table, current_column=column)